::: {.lead-finding}
With all 47 alternatives available, the closest loop used the same kit for all 48 references. When we excluded that kit, the exact kick, snare and hi-hat pattern ranked first for 13 of 48 references.
:::

Each cross-kit search has 36 candidates. Three play the reference performance. [Listen to the original comparisons](report.html).

In [ ]:
import csv
import json
from IPython.display import HTML, display
import plotly.graph_objects as go

with open("loop-similarities.csv", newline="") as source:
    reader = csv.reader(source)
    ids = next(reader)[1:]
    rows = list(reader)
assert [row[0] for row in rows] == ids
scores = [[float(value) for value in row[1:]] for row in rows]

# Load Plotly separately so Quarto keeps the figure with its section.
from plotly.offline import get_plotlyjs
display(HTML(f"<script>{get_plotlyjs()}</script>"))

## Explore five nearest neighbors

In [ ]:
controls = '''<div class="d-flex flex-wrap gap-3 mb-3">
  <label>Reference loop <select class="form-select" id="ref"></select></label>
  <label>Search <select class="form-select" id="scope">
    <option value="cross">Other kits only</option><option value="all">All kits</option>
  </select></label>
</div>
<ol id="rows" aria-live="polite"></ol>
'''
selector = r'''const names = {"black-pearl": "Black Pearl", "red-zeppelin": "Red Zeppelin", virtuosity: "Virtuosity", phazite: "Phazite"};
const loops = ids.map(id => {
  const [pattern, snare, kit] = id.split("_");
  const label = [names[kit], pattern.toUpperCase(), "snare " + [...snare].join(" & ")].join(" \u00b7 ");
  return {pattern, snare, kit, label};
});
const reference = document.querySelector("#ref");
const scope = document.querySelector("#scope");
loops.forEach((loop, i) => reference.add(new Option(loop.label, i)));
reference.value = ids.indexOf("p1_24_black-pearl");

function showNeighbors() {
  const i = Number(reference.value);
  const origin = loops[i];
  const ranked = loops.map((_, j) => j)
    .filter(j => j !== i && (scope.value === "all" || loops[j].kit !== origin.kit))
    .sort((a, b) => scores[i][b] - scores[i][a]);
  const rows = ranked.slice(0, 5).map(j => {
    const loop = loops[j];
    const match = loop.pattern === origin.pattern && loop.snare === origin.snare;
    const row = document.createElement("li");
    row.textContent = loop.label + " \u00b7 " + scores[i][j].toFixed(4) + (match ? " \u25cf" : "");
    return row;
  });
  document.querySelector("#rows").replaceChildren(...rows);
}
reference.onchange = scope.onchange = showNeighbors;
showNeighbors();

'''
data = json.dumps([ids, scores])
display(HTML(controls + '<script>{ const [ids, scores] = ' + data + ';' + selector + '}</script>'))

A filled circle marks an exact performance match. P1 through P4 identify the kick and hi-hat patterns. Higher cosine similarity means closer whole-loop embeddings.

## See all 48 loops at once

In [ ]:
from itertools import groupby

kit_names = {"black-pearl": "Black Pearl", "phazite": "Phazite",
             "red-zeppelin": "Red Zeppelin", "virtuosity": "Virtuosity"}
centers, labels, boundaries = [], [], []
start = 0
for kit, members in groupby(ids, key=lambda clip: clip.split("_")[-1]):
    end = start + len(list(members))
    centers.append((start + end - 1) / 2)
    labels.append(kit_names[kit])
    boundaries.append(end - 0.5)
    start = end

# Mask the self-comparisons so they do not dominate the diagonal.
values = [[None if i == j else value for j, value in enumerate(row)]
          for i, row in enumerate(scores)]
figure = go.Figure(go.Heatmap(
    z=values, colorscale="Viridis", zmax=1, hoverongaps=False,
    customdata=[[[row_id, col_id] for col_id in ids] for row_id in ids],
    hovertemplate="Reference: %{customdata[0]}<br>Comparison: %{customdata[1]}<br>Cosine: %{z:.3f}<extra></extra>",
    colorbar=dict(orientation="v", x=1.02, xanchor="left", y=0.5,
                  title=dict(text="Cosine similarity", side="right"), thickness=12, len=0.75),
))
ticks = dict(tickmode="array", tickvals=centers, ticktext=labels, fixedrange=True,
             showgrid=False, zeroline=False, automargin=True)
figure.update_xaxes(**ticks, side="top", constrain="domain", range=[-0.5, len(ids)-0.5], tickangle=-25)
figure.update_yaxes(**ticks, range=[len(ids)-0.5, -0.5], scaleanchor="x", constrain="domain")
# The masked diagonal reveals this matrix-sized background at every viewport width.
figure.add_shape(type="rect", x0=-0.5, y0=-0.5, x1=len(ids)-0.5, y1=len(ids)-0.5,
                 layer="below", fillcolor="#bbb", line_width=0)
for boundary in boundaries[:-1]:
    figure.add_vline(x=boundary, line_width=2, line_color="white")
    figure.add_hline(y=boundary, line_width=2, line_color="white")
figure.update_layout(height=590, margin=dict(l=90, r=10, t=65, b=45),
                     plot_bgcolor="#faf9f6", paper_bgcolor="#faf9f6")
# Plotly sizes colorbars against the full plotting area. Follow the square matrix
# as its vertical domain contracts on narrow screens.
align_legend = """
const chart = document.getElementById('{plot_id}');
function alignLegend() {
    const axis = chart._fullLayout.yaxis;
    const [bottom, top] = axis.domain;
    const length = top - bottom;
    const middle = (top + bottom) / 2;
    const angle = axis._length < 300 ? -70 : -25;
    if (chart.layout.xaxis.tickangle !== angle) {
        Plotly.relayout(chart, {'xaxis.tickangle': angle});
        return;
    }
    const bar = chart.data[0].colorbar;
    if (Math.abs(bar.len - length) > 0.001 || Math.abs(bar.y - middle) > 0.001) {
        Plotly.restyle(chart, {'colorbar.len': length, 'colorbar.y': middle}, [0]);
    }
}
chart.on('plotly_afterplot', alignLegend);
alignLegend();
"""
display(HTML(figure.to_html(full_html=False, include_plotlyjs=False, post_script=align_legend, config={"displayModeBar": False, "responsive": True})))

Pick the reference kit on the left and the comparison kit along the top. Each outlined block contains twelve performances per kit. Hover over a square for the two loop IDs and their cosine similarity.

The bright blocks along the main diagonal group recordings from each kit. Inside the other blocks, the line from top left to bottom right compares identical performances across kits. Black Pearl and Red Zeppelin share a closed hat. A controlled hat swap would help test how much that sound contributes to their similarity. Gray squares mark self-comparisons.